# 09 · Phase-Transformation Foundations

**Where this sits in PyTex.** Orientation-relationship (OR) analysis is PyTex's
flagship capability. This notebook introduces its *foundation primitives* — the
objects that later notebooks (18–21) turn into full identification, reconstruction,
and composite-diffraction workflows. We keep the scope deliberately conceptual: what
an `OrientationRelationship` **is**, how variants arise, and how a
`PhaseTransformationRecord` captures a transformation as an explainable,
provenance-carrying object.

## Learning goals

1. build an `OrientationRelationship` from a named crystallographic correspondence (Kurdjumov–Sachs, Nishiyama–Wassermann);
2. read its **self-describing** `describe()` output and its misorientation, checked against literature angles;
3. generate the crystallographic **variants** and count them;
4. examine the **intervariant misorientation spectrum**;
5. wrap parent + children into a `PhaseTransformationRecord`.

## Theory

When a parent phase transforms to a child phase, the two lattices are related by a
reproducible **orientation relationship** — a rotation fixed by parallel planes and
directions. For FCC→BCC steel the classical cases are

- **Kurdjumov–Sachs (K–S):** $(111)_\gamma \parallel (011)_\alpha$ and $[\bar101]_\gamma \parallel [\bar1\bar11]_\alpha$;
- **Nishiyama–Wassermann (N–W):** $(111)_\gamma \parallel (011)_\alpha$ and $[\bar211]_\gamma \parallel [0\bar11]_\alpha$.

Because the parent has cubic symmetry, one OR spawns a family of symmetry-equivalent
**variants** (24 for K–S, 12 for N–W in the classic counting). The set of misorientations
*between* variants — the **intervariant spectrum** — is a fingerprint used to identify
the OR from measured child data. The K–S misorientation from parent is $\approx42.85^\circ$
and N–W $\approx45.99^\circ$ about axes near $\langle hkl\rangle$ close to $\langle 011\rangle$
(Kurdjumov & Sachs 1930; Nishiyama 1934; Wassermann 1935; Morawiec 2004).

In [ ]:
from __future__ import annotations

import numpy as np

from pytex import (
    FrameDomain,
    Lattice,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    ReferenceFrame,
    SymmetrySpec,
    intervariant_misorientation_angles_deg,
)

np.set_printoptions(precision=3, suppress=True)

parent_crystal = ReferenceFrame("parent_crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
child_crystal = ReferenceFrame("child_crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
specimen = ReferenceFrame("specimen", FrameDomain.SPECIMEN, ("x", "y", "z"))

gamma_fcc = Phase(
    "gamma_fcc",
    lattice=Lattice(3.6, 3.6, 3.6, 90, 90, 90, crystal_frame=parent_crystal),
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=parent_crystal),
    crystal_frame=parent_crystal,
)
alpha_bcc = Phase(
    "alpha_bcc",
    lattice=Lattice(2.87, 2.87, 2.87, 90, 90, 90, crystal_frame=child_crystal),
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=child_crystal),
    crystal_frame=child_crystal,
)

## 1 · An orientation relationship that explains itself

`OrientationRelationship.from_kurdjumov_sachs_correspondence` builds the parent→child
rotation from the K–S correspondence. Following PyTex's **explainable-results
doctrine**, the object carries a `describe()` method that emits convention-explicit,
citation-ready scientific prose — the parallelisms, the misorientation representative,
and the variant count — computed live, not transcribed.

In [ ]:
ks = OrientationRelationship.from_kurdjumov_sachs_correspondence(
    parent_phase=gamma_fcc, child_phase=alpha_bcc,
)
print(ks.describe())

## 2 · Misorientation angles versus literature

The OR's misorientation from parent is a numerical fingerprint. K–S gives
$\approx42.85^\circ$ and N–W $\approx45.99^\circ$; we compute both and assert they match
the accepted values. These are the same rotations a parent-reconstruction routine
searches for in measured child orientations (Notebook 20).

In [ ]:
nw = OrientationRelationship.from_nishiyama_wassermann_correspondence(
    parent_phase=gamma_fcc, child_phase=alpha_bcc,
)
ks_angle = ks.misorientation().angle_deg
nw_angle = nw.misorientation().angle_deg
print(f"K-S parent->child misorientation: {ks_angle:.3f} deg  (literature ~42.85)")
print(f"N-W parent->child misorientation: {nw_angle:.3f} deg  (literature ~45.99)")
assert abs(ks_angle - 42.85) < 0.05
assert abs(nw_angle - 45.99) < 0.05

## 3 · Variants: one OR, many child orientations

Cubic parent symmetry means a single OR produces a family of crystallographically
distinct child orientations. `generate_variants()` enumerates them (24 for K–S). Each
`TransformationVariant` knows its index and its parent→child rotation — the building
blocks of variant-selection and composite-diffraction analysis.

In [ ]:
variants = ks.generate_variants()
print("number of K-S variants:", len(variants))
assert len(variants) == 24

first = variants[0]
w = abs(first.parent_to_child_rotation.quaternion[0])
raw_angle_deg = np.rad2deg(2.0 * np.arccos(np.clip(w, -1.0, 1.0)))
print("\nvariant 0:")
print("  variant_index         :", first.variant_index)
print("  parent operator index :", first.parent_operator_index)
print("  child operator index  :", first.child_operator_index)
print("  raw rotation angle deg:", round(raw_angle_deg, 3))

## 4 · The intervariant misorientation spectrum

`intervariant_misorientation_angles_deg` returns the full matrix of misorientation
angles between variants. Its distinct values are the **K–S variant angles** — a
signature (10.5°, 14.9°, 20.6°, 21.1°, 47.1°, 49.5°, 50.5°, 51.7°, 57.2°, 60°) that
appears in measured misorientation histograms of martensitic and bainitic
microstructures.

In [ ]:
angles = np.asarray(intervariant_misorientation_angles_deg(ks))
off_diagonal = angles[~np.eye(angles.shape[0], dtype=bool)]
distinct = np.unique(np.round(off_diagonal, 1))
print("intervariant angle matrix shape:", angles.shape)
print("distinct intervariant angles (deg):", distinct)
assert 60.0 in distinct  # the boundary self-misorientation is present

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.hist(off_diagonal, bins=np.arange(0, 63, 1.5), color="#2e86ab", edgecolor="white")
for value in distinct:
    ax.axvline(value, color="#d1495b", lw=0.8, alpha=0.6)
ax.set_xlabel("intervariant misorientation angle (deg)")
ax.set_ylabel("count of variant pairs")
ax.set_title("Kurdjumov-Sachs intervariant misorientation spectrum")
fig.tight_layout()


## 5 · Capturing a transformation as a record

A `PhaseTransformationRecord` bundles the parent orientation, the observed child
orientations, and the governing OR into one provenance-carrying object — the unit that
parent-reconstruction and variant-selection reports consume. Here we build the 24
predicted child orientations from a chosen parent (using the canonical
$C = P\,V^{\mathsf T}$ convention) and record them; `variant_count` confirms the family
size.

In [ ]:
parent = Orientation.from_euler(
    10.0, 20.0, 30.0, crystal_frame=parent_crystal, specimen_frame=specimen,
    symmetry=gamma_fcc.symmetry, phase=gamma_fcc,
)
P = parent.rotation.as_matrix()
child_matrices = np.stack(
    [P @ v.parent_to_child_rotation.as_matrix().T for v in variants]
)
children = OrientationSet.from_matrices(
    child_matrices, crystal_frame=child_crystal, specimen_frame=specimen,
    symmetry=alpha_bcc.symmetry, phase=alpha_bcc,
)
record = PhaseTransformationRecord(
    name="ks_demo",
    orientation_relationship=ks,
    parent_orientation=parent,
    child_orientations=children,
    variant_indices=np.array([v.variant_index for v in variants]),
)
print("record name        :", record.name)
print("governing OR        :", record.orientation_relationship.name)
print("variant_count       :", record.variant_count)
print("child orientations  :", len(record.child_orientations))
assert record.variant_count == 24

## Summary and where to go next

- An **`OrientationRelationship`** encodes a parent→child transformation via parallel
  planes/directions and **describes itself** with citation-ready prose.
- Its misorientation matches literature (**K–S $42.85^\circ$, N–W $45.99^\circ$**), and
  it spawns a **variant family** whose **intervariant spectrum** is a diagnostic fingerprint.
- A **`PhaseTransformationRecord`** captures parent + children + OR as one explainable,
  provenance-carrying object.

**Next:** the OR track goes deep from here —
[Notebook 18](18_orientation_relationships_fundamentals.ipynb) (fundamentals),
[Notebook 19](19_lattice_correspondence_and_transformation_strain.ipynb) (strain),
[Notebook 20](20_or_catalogs_identification_and_reconstruction.ipynb) (identification &
parent reconstruction), and [Notebook 21](21_composite_or_diffraction_patterns.ipynb)
(composite diffraction).